The script connects to a PostgreSQL database and uses the Open-Meteo API to retrieve current weather data based on a fixed geographic location (latitude and longitude). It then extracts relevant fields such as temperature and wind speed from the API response.

Once the data is retrieved, it is inserted into a structured table (live_weather) in the database along with a timestamp generated at runtime. This allows the dataset to grow over time and enables time-series analysis.

Finally, the script commits the transaction to ensure the data is saved and closes the database connection to maintain resource efficiency.

In [2]:
import requests
import psycopg2
import time
from datetime import datetime

API_URL = "https://api.open-meteo.com/v1/forecast"


conn = psycopg2.connect(
    dbname="weather_db",
    user="postgres",
    password="damian",
    host="localhost"
)

cur = conn.cursor()

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
params = {
    "latitude": 41.3839,
    "longitude": 2.1675,
    "current_weather": True
},
    timeout=10  
)

data = response.json()["current_weather"]

cur.execute("""
INSERT INTO live_weather (timestamp, temperature, windspeed)
VALUES (%s, %s, %s)
""", (
    datetime.now(),
    data["temperature"],
    data["windspeed"],
))

conn.commit()

cur.close()
conn.close()

print("Inserted successfully")

Inserted successfully
